# IBM-to-CRM Mapping Validation

This notebook checks the frozen Automotive 5 input contract without training, tuning, calibration, or test evaluation. It validates historical bands, controlled categories, product hierarchy, and illustrative Kenyan currency conversion. Actual probabilities are shown only if an approved, already-fitted artifact is present.

## 1. Frozen contract and artifact availability

The final model expects raw USD amount, integer history code, and exact IBM strings for route, group and subgroup. Notebook 08 retained its fitted pipelines only in kernel memory; this cell reports whether an approved export exists rather than refitting one.

In [1]:
from pathlib import Path
import json
import joblib

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
USD_PER_KES_RATE = 129  # Illustrative fixed demonstration rate, not a live exchange rate.
primary_columns = [
    'Opportunity Amount USD', 'Revenue From Client Past Two Years', 'Route To Market',
    'Supplies Group', 'Supplies Subgroup',
]
automotive_subgroups = [
    'Batteries & Accessories', 'Exterior Accessories', 'Garage & Car Care',
    'Interior Accessories', 'Replacement Parts', 'Towing & Hitches',
    'Car Electronics', 'Tires & Wheels', 'Motorcycle Parts', 'Shelters & RV',
]

repository_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
artifact_directory = repository_root / 'models' / 'automotive_v1'
primary_artifact_path = artifact_directory / 'automotive_5_primary.joblib'
fallback_artifact_path = artifact_directory / 'automotive_limited_history_fallback.joblib'
metadata_path = artifact_directory / 'metadata.json'
assert primary_artifact_path.exists() and fallback_artifact_path.exists() and metadata_path.exists()
primary_pipeline = joblib.load(primary_artifact_path)
fallback_pipeline = joblib.load(fallback_artifact_path)
metadata = json.loads(metadata_path.read_text(encoding='utf-8'))
assert metadata['primary_features'] == primary_columns
assert metadata['fallback_features'] == ['Opportunity Amount USD', 'Route To Market', 'Supplies Group', 'Supplies Subgroup']
assert list(primary_pipeline.named_steps['model'].classes_) == [0, 1]
print('Frozen Automotive 5 contract:', primary_columns)
print('Loaded artifact version:', metadata['model_version'])
print('Primary artifact:', primary_artifact_path.relative_to(repository_root))
print('Fallback artifact:', fallback_artifact_path.relative_to(repository_root))
print('Positive class:', metadata['probability_output'])

Frozen Automotive 5 contract: ['Opportunity Amount USD', 'Revenue From Client Past Two Years', 'Route To Market', 'Supplies Group', 'Supplies Subgroup']
Loaded artifact version: automotive_v1
Primary artifact: models/automotive_v1/automotive_5_primary.joblib
Fallback artifact: models/automotive_v1/automotive_limited_history_fallback.joblib
Positive class: raw Won probability; class 1 is Won


## 2. Recreate only the existing training membership

This repeats the fixed split membership so amount applicability can be described. It does not fit any preprocessing or model.

In [2]:
raw_df = pd.read_csv(repository_root / 'datasets/IBM-sales-opportunity.csv', keep_default_na=False, na_values=[''])
cleaned_df = raw_df.drop_duplicates().loc[lambda frame: ~frame['Opportunity Number'].duplicated(keep=False)].copy()
cleaned_df['target'] = cleaned_df['Opportunity Result'].map({'Won': 1, 'Loss': 0})

development_df, _ = train_test_split(cleaned_df, test_size=0.20, stratify=cleaned_df['target'], random_state=42)
fit_df, _ = train_test_split(development_df, test_size=0.25, stratify=development_df['target'], random_state=43)
training_df, _ = train_test_split(fit_df, test_size=1 / 6, stratify=fit_df['target'], random_state=44)
automotive_training_df = training_df.loc[training_df['Supplies Subgroup'].isin(automotive_subgroups)].copy()
amounts = automotive_training_df['Opportunity Amount USD']

display(pd.DataFrame({
    'USD amount': [amounts.min(), amounts.quantile(.05), amounts.quantile(.25), amounts.median(), amounts.quantile(.75), amounts.quantile(.95), amounts.max()],
}, index=['minimum', '5th percentile', '25th percentile', 'median', '75th percentile', '95th percentile', 'maximum']))
print(f'Zero amount share: {(amounts == 0).mean():.2%} ({(amounts == 0).sum():,} of {len(amounts):,})')
print(automotive_training_df[primary_columns].dtypes.to_string())

,USD amount
minimum,0.0
5th percentile,1221.0
25th percentile,15000.0
median,48000.0
75th percentile,105000.0
95th percentile,330861.0
maximum,1000000.0


Zero amount share: 2.55% (959 of 37,586)
Opportunity Amount USD                int64
Revenue From Client Past Two Years    int64
Route To Market                         str
Supplies Group                          str
Supplies Subgroup                       str


## 3. Source categories and hierarchy

These checks compare the original Automotive 5 training rows with the actual fitted `OneHotEncoder.categories_` loaded from the saved primary artifact.

In [3]:
primary_preprocessor = primary_pipeline.named_steps['preprocess']
primary_encoder = primary_preprocessor.named_transformers_['categorical']
primary_scaler = primary_preprocessor.named_transformers_['amount'].named_steps['scale']
fitted_categories = {column: values.tolist() for column, values in zip(primary_preprocessor.transformers_[0][2], primary_encoder.categories_)}
assert fitted_categories == metadata['primary_encoder_categories']
assert primary_scaler.n_features_in_ == 1
for column in primary_columns[1:]:
    print(f'{column}: {fitted_categories[column]}')

hierarchy = automotive_training_df[['Supplies Group', 'Supplies Subgroup']].drop_duplicates().sort_values(['Supplies Group', 'Supplies Subgroup'])
assert hierarchy['Supplies Subgroup'].nunique() == 10
assert hierarchy.groupby('Supplies Subgroup')['Supplies Group'].nunique().eq(1).all()
assert set(automotive_training_df['Route To Market']) == {'Fields Sales', 'Other', 'Reseller', 'Telecoverage', 'Telesales'}
assert set(automotive_training_df['Revenue From Client Past Two Years']) == {0, 1, 2, 3, 4}
assert fitted_categories['Revenue From Client Past Two Years'] == [0, 1, 2, 3, 4]
assert primary_preprocessor.get_feature_names_out().tolist() == metadata['primary_transformed_feature_order']
print('Fitted encoder categories, transformed feature order and numeric scaler are confirmed.')
display(hierarchy.reset_index(drop=True))

Revenue From Client Past Two Years: [0, 1, 2, 3, 4]
Route To Market: ['Fields Sales', 'Other', 'Reseller', 'Telecoverage', 'Telesales']
Supplies Group: ['Car Accessories', 'Car Electronics', 'Performance & Non-auto', 'Tires & Wheels']
Supplies Subgroup: ['Batteries & Accessories', 'Car Electronics', 'Exterior Accessories', 'Garage & Car Care', 'Interior Accessories', 'Motorcycle Parts', 'Replacement Parts', 'Shelters & RV', 'Tires & Wheels', 'Towing & Hitches']
Fitted encoder categories, transformed feature order and numeric scaler are confirmed.


,Supplies Group,Supplies Subgroup
0,Car Accessories,Batteries & Accessories
1,Car Accessories,Exterior Accessories
2,Car Accessories,Garage & Car Care
3,Car Accessories,Interior Accessories
4,Car Accessories,Replacement Parts
5,Car Accessories,Towing & Hitches
6,Car Electronics,Car Electronics
7,Performance & Non-auto,Motorcycle Parts
8,Performance & Non-auto,Shelters & RV
9,Tires & Wheels,Tires & Wheels


## 4. Revenue-band and hierarchy validation

The mapper keeps verified zero, positive revenue, and unavailable history distinct. The historical dictionary does not define a code for a positive amount below USD 1, so that state is flagged instead of guessed.

In [4]:
def revenue_code(usd_revenue, history_complete=True):
    if not history_complete or pd.isna(usd_revenue):
        return None
    if usd_revenue < 0:
        raise ValueError('Historical revenue cannot be negative.')
    if usd_revenue == 0:
        return 0
    if usd_revenue < 1:
        return 'undefined positive amount below USD 1'
    if usd_revenue < 50_000:
        return 1
    if usd_revenue < 400_000:
        return 2
    if usd_revenue < 1_500_000:
        return 3
    return 4

boundary_values = [0, 1, 49_999.99, 50_000, 399_999.99, 400_000, 1_499_999.99, 1_500_000, .50]
boundary_results = pd.DataFrame({'USD revenue': boundary_values, 'assigned code': [revenue_code(value) for value in boundary_values]})
display(boundary_results)
assert boundary_results['assigned code'].tolist() == [0, 1, 1, 2, 2, 3, 3, 4, 'undefined positive amount below USD 1']

hierarchy_map = hierarchy.set_index('Supplies Subgroup')['Supplies Group'].to_dict()
assert hierarchy_map['Motorcycle Parts'] == 'Performance & Non-auto'
assert hierarchy_map['Replacement Parts'] == 'Car Accessories'
print('Revenue boundaries and selected hierarchy pairs passed.')

,USD revenue,assigned code
0,0.00,0
1,1.00,1
2,49999.99,1
3,50000.00,2
4,399999.99,2
5,400000.00,3
6,1499999.99,3
7,1500000.00,4
8,0.50,undefined positive amount below USD 1


Revenue boundaries and selected hierarchy pairs passed.


## 5. Illustrative Kenyan inputs, transformations and real inference

KES values use the fixed illustrative rate of 129 KES/USD. Currency conversion produces the raw USD input; revenue-band assignment produces an integer history category; the loaded pipeline itself then one-hot encodes categories and applies `log1p` plus the fitted scaler to amount before Random Forest inference. The adapter must not manually transform or scale the raw USD amount. Percentile is only the share of original Automotive 5 training amounts at or below the converted amount; it is not a chance of winning.

In [5]:
cases = pd.DataFrame([
    ['A: verified no history', 250_000, 0, True, 'Reseller', 'Car Accessories', 'Replacement Parts'],
    ['B: lower revenue', 1_000_000, 800_000, True, 'Fields Sales', 'Car Accessories', 'Batteries & Accessories'],
    ['C: higher revenue', 10_000_000, 8_000_000, True, 'Reseller', 'Car Accessories', 'Exterior Accessories'],
    ['D: history unavailable', 500_000, np.nan, False, 'Telesales', 'Car Accessories', 'Garage & Car Care'],
], columns=['case', 'amount KES', 'prior revenue KES', 'history complete', 'Route To Market', 'Supplies Group', 'Supplies Subgroup'])
cases['Opportunity Amount USD'] = cases['amount KES'] / USD_PER_KES_RATE
history_codes = [revenue_code(value / USD_PER_KES_RATE, complete) for value, complete in zip(cases['prior revenue KES'], cases['history complete'])]
cases['Revenue From Client Past Two Years'] = pd.Series(history_codes, index=cases.index, dtype='object')
cases['training amount percentile'] = [(amounts <= value).mean() for value in cases['Opportunity Amount USD']]
cases['inference route'] = np.where(cases['history complete'], 'primary if artifact is available', 'limited-history fallback if its artifact is available')

def validate_for_inference(row):
    if pd.isna(row['Opportunity Amount USD']):
        return 'reject: amount is unknown'
    if row['Opportunity Amount USD'] < 0:
        return 'reject: amount is negative'
    if row['Route To Market'] not in set(automotive_training_df['Route To Market']):
        return 'reject: unsupported route'
    if row['Supplies Subgroup'] not in hierarchy_map:
        return 'reject: unsupported subgroup'
    if hierarchy_map[row['Supplies Subgroup']] != row['Supplies Group']:
        return 'reject: inconsistent group/subgroup'
    history_code = row['Revenue From Client Past Two Years']
    if pd.isna(history_code):
        return 'fallback: history unavailable'
    if not isinstance(history_code, (int, np.integer)) or history_code not in {0, 1, 2, 3, 4}:
        return 'reject: invalid or undefined history code'
    return 'primary: compatible input'

cases['validation result'] = cases.apply(validate_for_inference, axis=1)

assert cases['validation result'].tolist() == ['primary: compatible input', 'primary: compatible input', 'primary: compatible input', 'fallback: history unavailable']
unsupported = cases.iloc[0].copy(); unsupported['Supplies Subgroup'] = 'Unrepresented Product'
invalid_pair = cases.iloc[0].copy(); invalid_pair['Supplies Subgroup'] = 'Motorcycle Parts'
unknown_amount = cases.iloc[0].copy(); unknown_amount['Opportunity Amount USD'] = np.nan
assert validate_for_inference(unsupported) == 'reject: unsupported subgroup'
assert validate_for_inference(invalid_pair) == 'reject: inconsistent group/subgroup'
assert validate_for_inference(unknown_amount) == 'reject: amount is unknown'
undefined_history = cases.iloc[0].copy(); undefined_history['Revenue From Client Past Two Years'] = revenue_code(.50)
string_history = cases.iloc[0].copy(); string_history['Revenue From Client Past Two Years'] = '1'
invalid_history = cases.iloc[0].copy(); invalid_history['Revenue From Client Past Two Years'] = 5
assert validate_for_inference(undefined_history) == 'reject: invalid or undefined history code'
assert validate_for_inference(string_history) == 'reject: invalid or undefined history code'
assert validate_for_inference(invalid_history) == 'reject: invalid or undefined history code'

primary_rows = cases.loc[cases['validation result'] == 'primary: compatible input'].copy()
primary_input = primary_rows[primary_columns].copy()
# Keep the history code as an integer so it matches the fitted encoder categories 0–4.
primary_input['Revenue From Client Past Two Years'] = primary_input['Revenue From Client Past Two Years'].astype('int64')
primary_rows['model used'] = 'Automotive 5 primary'
primary_rows['predicted Won probability'] = primary_pipeline.predict_proba(primary_input)[:, 1]

fallback_rows = cases.loc[cases['validation result'] == 'fallback: history unavailable'].copy()
fallback_columns = metadata['fallback_features']
fallback_input = fallback_rows[fallback_columns].copy()
fallback_rows['model used'] = 'Automotive limited-history fallback'
fallback_rows['predicted Won probability'] = fallback_pipeline.predict_proba(fallback_input)[:, 1]

scored_cases = pd.concat([primary_rows, fallback_rows], ignore_index=True).sort_values('case')
assert scored_cases['predicted Won probability'].between(0, 1).all()
display(scored_cases[[
    'case', 'amount KES', 'Opportunity Amount USD', 'prior revenue KES',
    'Revenue From Client Past Two Years', 'Route To Market', 'Supplies Group',
    'Supplies Subgroup', 'model used', 'predicted Won probability',
    'training amount percentile', 'validation result',
]].round({'Opportunity Amount USD': 2, 'predicted Won probability': 5, 'training amount percentile': 4}))

# This display proves that raw CRM values, not manually scaled values, enter the fitted preprocessor.
display(pd.DataFrame(primary_preprocessor.transform(primary_input), columns=primary_preprocessor.get_feature_names_out()).head())
print('Unsupported product, invalid hierarchy, unknown amount and invalid history codes are rejected; unavailable history routes to fallback.')
print(f'Confirmed zero amount exists in training data: {(amounts == 0).any()}; a missing amount remains invalid.')
print('Rejected cases never reach predict_proba; Case D reaches only the fallback pipeline.')

,case,amount KES,Opportunity Amount USD,prior revenue KES,Revenue From Client Past Two Years,Route To Market,Supplies Group,Supplies Subgroup,model used,predicted Won probability,training amount percentile,validation result
0,A: verified no history,250000,1937.98,0.0,0,Reseller,Car Accessories,Replacement Parts,Automotive 5 primary,0.38613,0.0577,primary: compatible input
1,B: lower revenue,1000000,7751.94,800000.0,1,Fields Sales,Car Accessories,Batteries & Accessories,Automotive 5 primary,0.85528,0.1380,primary: compatible input
2,C: higher revenue,10000000,77519.38,8000000.0,2,Reseller,Car Accessories,Exterior Accessories,Automotive 5 primary,0.61912,0.6462,primary: compatible input
3,D: history unavailable,500000,3875.97,NaN,None,Telesales,Car Accessories,Garage & Car Care,Automotive limited-history fallback,0.50894,0.0817,fallback: history unavailable


,categorical__Revenue From Client Past Two Years_0,categorical__Revenue From Client Past Two Years_1,categorical__Revenue From Client Past Two Years_2,categorical__Revenue From Client Past Two Years_3,categorical__Revenue From Client Past Two Years_4,categorical__Route To Market_Fields Sales,categorical__Route To Market_Other,categorical__Route To Market_Reseller,categorical__Route To Market_Telecoverage,categorical__Route To Market_Telesales,...,categorical__Supplies Subgroup_Car Electronics,categorical__Supplies Subgroup_Exterior Accessories,categorical__Supplies Subgroup_Garage & Car Care,categorical__Supplies Subgroup_Interior Accessories,categorical__Supplies Subgroup_Motorcycle Parts,categorical__Supplies Subgroup_Replacement Parts,categorical__Supplies Subgroup_Shelters & RV,categorical__Supplies Subgroup_Tires & Wheels,categorical__Supplies Subgroup_Towing & Hitches,amount__Opportunity Amount USD
0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,-1.205985
1,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,-0.595885
2,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.417702


Unsupported product, invalid hierarchy, unknown amount and invalid history codes are rejected; unavailable history routes to fallback.
Confirmed zero amount exists in training data: True; a missing amount remains invalid.
Rejected cases never reach predict_proba; Case D reaches only the fallback pipeline.


## 6. Small Kenyan B2B Opportunity Compatibility Test

Each comparison changes only opportunity amount. The route and product remain Reseller / Car Accessories / Replacement Parts. The first scenario has verified zero history; the second has verified KES 800,000 prior revenue. Percentiles and bands describe IBM training-input support, not Kenyan win chances.

In [6]:
amount_band_edges = [-np.inf, 0, 500, 1_000, 5_000, 10_000, np.inf]
amount_band_labels = ['USD 0', 'USD >0–500', 'USD >500–1,000', 'USD >1,000–5,000', 'USD >5,000–10,000', 'USD >10,000']
training_amount_bands = pd.cut(amounts, bins=amount_band_edges, labels=amount_band_labels)
amount_support = automotive_training_df.assign(amount_band=training_amount_bands).groupby('amount_band', observed=False).agg(
    training_opportunities=('target', 'size'),
    training_won_rate=('target', 'mean'),
).reindex(amount_band_labels)
amount_support['training_percentage'] = amount_support['training_opportunities'] / len(automotive_training_df)
display(amount_support.reset_index().round({'training_won_rate': 4, 'training_percentage': 4}))

small_amounts_kes = [25_000, 50_000, 100_000, 250_000, 500_000, 1_000_000]
def support_band(usd_amount):
    return pd.cut(pd.Series([usd_amount]), bins=amount_band_edges, labels=amount_band_labels).iloc[0]

def score_controlled_amounts(history_name, prior_revenue_kes):
    history_code = revenue_code(prior_revenue_kes / USD_PER_KES_RATE)
    assert isinstance(history_code, int) and history_code in {0, 1, 2, 3, 4}
    records = pd.DataFrame({
        'Opportunity Amount USD': [amount / USD_PER_KES_RATE for amount in small_amounts_kes],
        'Revenue From Client Past Two Years': [history_code] * len(small_amounts_kes),
        'Route To Market': ['Reseller'] * len(small_amounts_kes),
        'Supplies Group': ['Car Accessories'] * len(small_amounts_kes),
        'Supplies Subgroup': ['Replacement Parts'] * len(small_amounts_kes),
    })
    assert records.apply(validate_for_inference, axis=1).eq('primary: compatible input').all()
    records['amount KES'] = small_amounts_kes
    records['history scenario'] = history_name
    records['model used'] = 'Automotive 5 primary'
    records['training amount percentile'] = [(amounts <= value).mean() for value in records['Opportunity Amount USD']]
    records['training amount band'] = [support_band(value) for value in records['Opportunity Amount USD']]
    records['training observations in band'] = records['training amount band'].map(amount_support['training_opportunities']).astype(int)
    records['predicted Won probability'] = primary_pipeline.predict_proba(records[primary_columns])[:, 1]
    return records

zero_history_amounts = score_controlled_amounts('verified KES 0 history', 0)
positive_history_amounts = score_controlled_amounts('verified KES 800,000 history', 800_000)
small_amount_scores = pd.concat([zero_history_amounts, positive_history_amounts], ignore_index=True)
assert small_amount_scores['predicted Won probability'].between(0, 1).all()
display(small_amount_scores[[
    'history scenario', 'amount KES', 'Opportunity Amount USD', 'Revenue From Client Past Two Years',
    'training amount percentile', 'training amount band', 'training observations in band',
    'model used', 'predicted Won probability',
]].round({'Opportunity Amount USD': 2, 'training amount percentile': 4, 'predicted Won probability': 5}))

fallback_50k_input = pd.DataFrame([{
    'Opportunity Amount USD': 50_000 / USD_PER_KES_RATE,
    'Route To Market': 'Reseller',
    'Supplies Group': 'Car Accessories',
    'Supplies Subgroup': 'Replacement Parts',
}])
fallback_50k_probability = fallback_pipeline.predict_proba(fallback_50k_input[fallback_columns])[:, 1][0]
assert np.isfinite(fallback_50k_probability) and 0 <= fallback_50k_probability <= 1
display(pd.DataFrame([{
    'case': 'KES 50,000 with unavailable history',
    'Opportunity Amount USD': 50_000 / USD_PER_KES_RATE,
    'training amount percentile': (amounts <= (50_000 / USD_PER_KES_RATE)).mean(),
    'model used': 'Automotive limited-history fallback',
    'predicted Won probability': fallback_50k_probability,
}]).round({'Opportunity Amount USD': 2, 'training amount percentile': 4, 'predicted Won probability': 5}))
print('KES 50,000 is USD 387.60: represented by the USD >0–500 training band, not automatically rare or invalid.')

,amount_band,training_opportunities,training_won_rate,training_percentage
0,USD 0,959,0.1543,0.0255
1,USD >0–500,484,0.4091,0.0129
2,"USD >500–1,000",350,0.4657,0.0093
3,"USD >1,000–5,000",2328,0.4287,0.0619
4,"USD >5,000–10,000",3372,0.3200,0.0897
5,"USD >10,000",30093,0.1987,0.8006


,history scenario,amount KES,Opportunity Amount USD,Revenue From Client Past Two Years,training amount percentile,training amount band,training observations in band,model used,predicted Won probability
0,verified KES 0 history,25000,193.80,0,0.0340,USD >0–500,484,Automotive 5 primary,0.53430
1,verified KES 0 history,50000,387.60,0,0.0367,USD >0–500,484,Automotive 5 primary,0.55222
2,verified KES 0 history,100000,775.19,0,0.0406,"USD >500–1,000",350,Automotive 5 primary,0.52722
3,verified KES 0 history,250000,1937.98,0,0.0577,"USD >1,000–5,000",2328,Automotive 5 primary,0.38613
4,verified KES 0 history,500000,3875.97,0,0.0817,"USD >1,000–5,000",2328,Automotive 5 primary,0.50584
5,verified KES 0 history,1000000,7751.94,0,0.1380,"USD >5,000–10,000",3372,Automotive 5 primary,0.37199
6,"verified KES 800,000 history",25000,193.80,1,0.0340,USD >0–500,484,Automotive 5 primary,0.90459
7,"verified KES 800,000 history",50000,387.60,1,0.0367,USD >0–500,484,Automotive 5 primary,0.90618
8,"verified KES 800,000 history",100000,775.19,1,0.0406,"USD >500–1,000",350,Automotive 5 primary,0.91434
9,"verified KES 800,000 history",250000,1937.98,1,0.0577,"USD >1,000–5,000",2328,Automotive 5 primary,0.91997


,case,Opportunity Amount USD,training amount percentile,model used,predicted Won probability
0,"KES 50,000 with unavailable history",387.6,0.0367,Automotive limited-history fallback,0.59595


KES 50,000 is USD 387.60: represented by the USD >0–500 training band, not automatically rare or invalid.


## 7. Result

The mapping checks pass for documented bands, loaded encoder categories, product hierarchy, known-zero versus missing amount/history treatment, and saved-artifact inference. The saved models return raw Won probabilities; no score is a commercial-priority decision. These technically valid predictions do not validate calibration or transfer to Kenyan business data.